# Cold Start SFT Data Preprocessing — Google Colab
**Preprocesses Sky-T1 and OpenThoughts CoT datasets and pushes to Hugging Face Dataset Hub**

This notebook runs in Google Colab. It cleans raw CoT datasets, standardizes thinking `<think>...</think>` and answer `<answer>...</answer>` tags, and uploads the formatted dataset to Hugging Face as `abhinav0231/reasoning-cold-start-sft-data`.

## Cell 1 — Install Dependencies & Authentication

In [ ]:
!pip install datasets huggingface_hub -q

import os
from huggingface_hub import login

# Hugging Face Authentication via Colab secrets or environment variable
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', 'YOUR_HF_TOKEN_HERE')

login(token=HF_TOKEN)
print('✅ Authenticated with Hugging Face Hub')

## Cell 2 — Configuration

In [ ]:
HF_USERNAME = "abhinav0231"
OUTPUT_DATASET_REPO = f"{HF_USERNAME}/reasoning-cold-start-sft-data"

MAX_SAMPLES_SKYT1       = 2500    # Sky-T1: diverse (math + code + logic)
MAX_SAMPLES_OPENTHOUGHT = 1500    # OpenThoughts: hard math CoT
SEED                    = 42

SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

print(f"Target HF Dataset Repo : {OUTPUT_DATASET_REPO}")
print(f"Sky-T1 Max Samples     : {MAX_SAMPLES_SKYT1}")
print(f"OpenThoughts Max Samples: {MAX_SAMPLES_OPENTHOUGHT}")

## Cell 3 — Data Cleaning Logic

In [ ]:
import re

def wrap_answer(text):
    text = text.strip()
    if "<answer>" in text:
        return text
    boxed = re.search(r"\\boxed\{(.+?)\}", text)
    ans = boxed.group(1) if boxed else text[-400:]
    return f"<answer>{ans}</answer>"

# ── Sky-T1 Cleaning ───────────────────────────────────────────────────────────
def clean_sky_t1(example):
    try:
        convs = example.get("conversations", [])
        user_msg = next((c["value"] for c in convs if c["from"] == "user"), None)
        asst_msg = next((c["value"] for c in convs if c["from"] == "assistant"), None)
        if not user_msg or not asst_msg:
            return None
        asst_msg = asst_msg.strip()
        if len(asst_msg) < 100:   # too short to be real CoT
            return None
        
        final_ans_match = re.search(
            r"(?:final answer|therefore,? the answer is|the answer is)[:\s]+(.+?)(?:\n|$)",
            asst_msg, re.IGNORECASE | re.DOTALL
        )
        if final_ans_match:
            ans_text    = final_ans_match.group(1).strip()[:300]
            think_body  = asst_msg[:final_ans_match.start()].strip()
        else:
            paragraphs = [p.strip() for p in asst_msg.split("\n\n") if p.strip()]
            ans_text   = paragraphs[-1][:300] if paragraphs else asst_msg[-300:]
            think_body = "\n\n".join(paragraphs[:-1]) if len(paragraphs) > 1 else asst_msg

        answer_part    = wrap_answer(ans_text)
        assistant_text = f"<think>\n{think_body}\n</think>\n{answer_part}"

        return {"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": assistant_text},
        ]}
    except:
        return None

# ── OpenThoughts Cleaning ─────────────────────────────────────────────────────
def clean_openthoughts(example):
    try:
        msgs     = example.get("messages", [])
        user_msg = next((m["content"] for m in msgs if m["role"] == "user"), None)
        asst_msg = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
        if not user_msg or not asst_msg or len(asst_msg.strip()) < 100:
            return None
        asst_msg = asst_msg.strip()
        if "<think>" in asst_msg and "</think>" in asst_msg:
            think   = re.search(r"<think>(.*?)</think>", asst_msg, re.DOTALL).group(1)
            after   = asst_msg.split("</think>", 1)[-1].strip()
            after   = wrap_answer(after)
            assistant_text = f"<think>{think}</think>\n{after}"
        else:
            paragraphs = [p.strip() for p in asst_msg.split("\n\n") if p.strip()]
            ans_text   = paragraphs[-1][:300] if paragraphs else asst_msg[-300:]
            think_body = "\n\n".join(paragraphs[:-1]) if len(paragraphs) > 1 else asst_msg
            assistant_text = f"<think>\n{think_body}\n</think>\n{wrap_answer(ans_text)}"
        return {"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": assistant_text},
        ]}
    except:
        return None

print("✅ Cleaning functions ready")

## Cell 4 — Download, Process, and Merge Datasets

In [ ]:
from datasets import load_dataset, Dataset
import random

# ── Load & Clean Sky-T1 ───────────────────────────────────────────────────────
print("Loading NovaSky-AI/Sky-T1_data_17k...")
sky_raw = load_dataset("NovaSky-AI/Sky-T1_data_17k", split="train").shuffle(seed=SEED)
sky_cleaned = []
for i, ex in enumerate(sky_raw):
    out = clean_sky_t1(ex)
    if out:
        sky_cleaned.append(out)
    if len(sky_cleaned) >= MAX_SAMPLES_SKYT1:
        break
print(f"  Sky-T1 → {len(sky_cleaned)} samples processed")

# ── Load & Clean OpenThoughts ─────────────────────────────────────────────────
print("Loading open-r1/OpenThoughts-114k-math...")
ot_raw = load_dataset("open-r1/OpenThoughts-114k-math", split="train").shuffle(seed=SEED)
ot_cleaned = []
for i, ex in enumerate(ot_raw):
    out = clean_openthoughts(ex)
    if out:
        ot_cleaned.append(out)
    if len(ot_cleaned) >= MAX_SAMPLES_OPENTHOUGHT:
        break
print(f"  OpenThoughts → {len(ot_cleaned)} samples processed")

# ── Combine & Shuffle ─────────────────────────────────────────────────────────
all_samples = sky_cleaned + ot_cleaned
random.seed(SEED)
random.shuffle(all_samples)

print(f"\n✅ Merged Total: {len(all_samples)} samples")
dataset = Dataset.from_list(all_samples)
print(f"Dataset structure: {dataset}")

## Cell 5 — Push Preprocessed Dataset to Hugging Face Hub

In [ ]:
print(f"Pushing dataset to Hugging Face: {OUTPUT_DATASET_REPO} ...")
dataset.push_to_hub(OUTPUT_DATASET_REPO, private=False)
print(f"✅ Preprocessed dataset successfully pushed to:")
print(f"   https://huggingface.co/datasets/{OUTPUT_DATASET_REPO}")